In [ ]:
import sys, os
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))  # scrape_code/
sys.path.insert(0, os.path.join(_ROOT, 'scrapers'))
sys.path.insert(0, os.path.join(_ROOT, 'utils'))

In [1]:
import numpy as np 
import pandas as pd 
from scraper import scrape_model 
import os, sys, time, json, re
import requests 
import datetime, dateparser
from bs4 import BeautifulSoup
import base64
import brotli, gzip, zlib 

from selenium import webdriver 
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By 
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

test_proxies = {
    'http': 'http://proxy_ip:port',
    'https': 'http://proxy_ip:port'
}


def content_parse_func(content):
    try:
        if content is not None:
            row_decoded = base64.b64decode(content)
            row_decompress = zlib.decompress(row_decoded, 16+zlib.MAX_WBITS)
            try:
                decode_row = row_decompress.decode('utf-8')
            except Exception as e:
                decode_row = row_decompress.decode('utf-8', errors='ignore')
            try:
                row_json = json.loads(decode_row)
                return row_json 
            except Exception as e:
                return None 
    except:
        return None 
    

def decode_func(val):
    decoded = base64.b64decode(val)
    brot_decode = brotli.decompress(decoded)
    return decoded, brot_decode


def close_out_driver(input_driver):
    input_driver.close()
    input_driver.quit()
    
            

urc_link = "world/united-rugby-championship-"
prem_link = "england/premiership-rugby-"
t14_link = "france/top-14-"
six_nations = "europe/six-nations-"
champ_cup = "europe/european-rugby-champions-cup-"

input_season = '2024-2025'
input_league = urc_link

test_url = "https://www.oddsportal.com/rugby-union/{}/results/".format(input_league+input_season)
print(test_url)


headers = {'User-Agent': 
           'Mozilla/5.0 (X11; Linux x86_64)'+\
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument('--disable-dev-shm-usage')
options.add_argument("user-agent={}".format(headers['User-Agent']))
driver = webdriver.Chrome(options=options)

driver.get(test_url)



https://www.oddsportal.com/rugby-union/world/united-rugby-championship-2024-2025/results/


In [11]:
driver.close()
driver.quit()

In [2]:
# class_check = driver.find_element(by=By.ID, value='ltauYFrA')
# class_check.find_elements(by=By.TAG_NAME, value='div')[2].text
# class_check_v2 = driver.find_element(by=By.ID, value="app")

core_vals = driver.find_elements(by=By.CSS_SELECTOR, value="[class='flex flex-col px-3 text-sm max-mm:px-0']")[0]
core_val_elem = core_vals.text

if core_val_elem != '':
    game_rows = core_vals.find_elements(by=By.CSS_SELECTOR, value="[class='eventRow flex w-full flex-col text-xs'")
else:
    print('Re-Run, missed data')

In [3]:
def parse_game_row(game_row):
    gr_dict = {}
    gr_split = game_row.text.split('\n')

    starter = len(gr_split)-10
    return starter, gr_split
    
    # gr_dict['game_date'] = gr_split[starter-5]
    # gr_dict['game_time'] = 
    # gr_dict['home_team'], gr_dict['home_score'] = 
    # gr_dict['away_team'], gr_dict['away_score'] =
    # gr_dict['home_odds'], gr_dict['away_odds'] = 
    # gr_dict['tie_odds'], gr_dict['bettors_offered'] =   



In [4]:
test_start, test_row = parse_game_row(game_rows[0])

test_row

['Rugby Union',
 '/',
 'World',
 '/',
 'United Rugby Championship 2024/2025',
 '14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4']

In [5]:
test_row[test_start-5:]

['14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4']

In [6]:
test_start_2, test_row_2 = parse_game_row(game_rows[1])

In [7]:
test_start_3, test_row_3 = parse_game_row(game_rows[2])

In [8]:
test_row_3

['09:45',
 'Leinster',
 '37',
 '–',
 '19',
 'Glasgow Warriors',
 '-714',
 '+2800',
 '+450',
 '4']

In [9]:
test_row_3[test_start_3:]

['09:45',
 'Leinster',
 '37',
 '–',
 '19',
 'Glasgow Warriors',
 '-714',
 '+2800',
 '+450',
 '4']

In [12]:
test_row_2[test_start_2-5:]

['07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4']

In [10]:
test_row_2

['07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4']

In [6]:
test_start

10

In [7]:
test_row[test_start:]

['12:00', 'Leinster', '32', '–', '7', 'Bulls', '-556', '+2800', '+392', '4']

In [12]:
test_row[test_start:]

['12:00', 'Leinster', '32', '–', '7', 'Bulls', '-556', '+2800', '+392', '4']

In [13]:
len(game_rows[0].text.split('\n'))

20

In [8]:
game_rows[0].text.split('\n')

['Rugby Union',
 '/',
 'World',
 '/',
 'United Rugby Championship 2024/2025',
 '14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4']

In [20]:
game_rows[3].text.split('\n')

['31 May 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:30',
 'Sharks',
 '25',
 '–',
 '24',
 'Munster',
 'pen.',
 '-263',
 '+2150',
 '+211',
 '4']

In [19]:
game_rows[1].text.split('\n')

['07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4']

In [14]:
lst_arrs = [len(i.text.split('\n')) for i in game_rows]

In [15]:
lst_arrs.find(16)

AttributeError: 'list' object has no attribute 'find'

In [13]:
set([len(i.text.split('\n')) for i in game_rows])

{10, 15, 16, 20}

In [11]:
game_rows[6].text.split('\n')

['30 May 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '14:35',
 'Glasgow Warriors',
 '36',
 '–',
 '18',
 'Stormers',
 '-250',
 '+2025',
 '+204',
 '4']

In [9]:
game_rows[3].text.split('\n')

['31 May 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:30',
 'Sharks',
 '25',
 '–',
 '24',
 'Munster',
 'pen.',
 '-263',
 '+2150',
 '+211',
 '4']

In [10]:
game_rows[0].text.split('\n')

['Rugby Union',
 '/',
 'World',
 '/',
 'United Rugby Championship 2024/2025',
 '14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4']

In [11]:
game_rows[1].text.split('\n')

['07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4']

In [12]:
game_rows[10].text.split('\n')

['11:15', 'Lions', '29', '–', '28', 'Ospreys', '-238', '+2100', '+188', '4']

In [30]:
len(game_rows[0].text.split('\n'))

20

In [31]:
len(game_rows[1].text.split('\n'))

15

In [33]:
len(game_rows[12].text.split('\n'))

15

In [34]:
len(game_rows[10].text.split('\n'))

10

In [29]:
game_rows[0].text.split('\n')

['Rugby Union',
 '/',
 'World',
 '/',
 'United Rugby Championship 2024/2025',
 '14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4']

In [22]:
game_rows[1].text.split('\n')

['07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4']

In [27]:
game_rows[10].text.split('\n')

['11:15', 'Lions', '29', '–', '28', 'Ospreys', '-238', '+2100', '+188', '4']

In [25]:
game_rows[12].text.split('\n')

['16 May 2025',
 '1',
 'X',
 '2',
 "B's",
 '15:00',
 'Munster',
 '30',
 '–',
 '21',
 'Benetton',
 '-500',
 '+2750',
 '+350',
 '4']

In [18]:
len(game_rows)

30

In [20]:
core_val_elem.split('\n')

['Rugby Union',
 '/',
 'World',
 '/',
 'United Rugby Championship 2024/2025',
 '14 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:00',
 'Leinster',
 '32',
 '–',
 '7',
 'Bulls',
 '-556',
 '+2800',
 '+392',
 '4',
 '07 Jun 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:15',
 'Bulls',
 '25',
 '–',
 '13',
 'Sharks',
 '-333',
 '+2225',
 '+260',
 '4',
 '09:45',
 'Leinster',
 '37',
 '–',
 '19',
 'Glasgow Warriors',
 '-714',
 '+2800',
 '+450',
 '4',
 '31 May 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '12:30',
 'Sharks',
 '25',
 '–',
 '24',
 'Munster',
 'pen.',
 '-263',
 '+2150',
 '+211',
 '4',
 '10:00',
 'Leinster',
 '33',
 '–',
 '21',
 'Scarlets',
 '-10000',
 '+4000',
 '+1200',
 '4',
 '07:30',
 'Bulls',
 '42',
 '–',
 '33',
 'Edinburgh',
 '-1429',
 '+3325',
 '+688',
 '4',
 '30 May 2025 - Play Offs',
 '1',
 'X',
 '2',
 "B's",
 '14:35',
 'Glasgow Warriors',
 '36',
 '–',
 '18',
 'Stormers',
 '-250',
 '+2025',
 '+204',
 '4',
 '17 May 2025',
 '1',
 'X',
 '2',
 "B's",
 '14:35',
 'Leinste

In [5]:
[i.text for i in class_check_v2.find_elements(by=By.TAG_NAME, value='div')]

['Odds formats:\nMoney Line Odds\nLanguage:\nEnglish\nLOGIN\nREGISTER\nODDS COMPARISON\nCOMMUNITY\nBOOKMAKERS\nBet $5 and Get $150 in Bonus Bets if You Win!\nCLAIM\nHOME\nNEXT MATCHES\nDROPPING ODDS\nSURE BETS\nIN PLAY ODDS\nALL EVENTS\nBETTING TOOLS\nBETTING\nMY LEAGUES\n0\nManage my leagues\nFOOTBALL\nBASKETBALL\nTENNIS\nBASEBALL\nHOCKEY\nAMERICAN FOOTBALL\nAUSSIE RULES\nBADMINTON\nBANDY\nBOXING\nCRICKET\nDARTS\nESPORTS\nFLOORBALL\nFUTSAL\nHANDBALL\nMMA\nRUGBY LEAGUE\nRUGBY UNION\nSNOOKER\nVOLLEYBALL\nGet up to $1,500 paid back\nin bonus bets, if you don`t\nwin !\nCLAIM\nBet $5 and Get $150 in\nBonus Bets if You Win!\nCLAIM\nChoose: First Bet Safety Net or\nBet $5 & Get $150\nCLAIM\nHome\n>\nRugby Union\n>\nWorld\n>\nUnited Rugby Championship 2024/2025\n>\nResults\nUnited Rugby Championship 2024/2025 Results, Scores & Historical Odds\nNext Matches\nResults\nStandings\n2025/2026\n2024/2025\n2023/2024\n2022/2023\n2021/2022\n2020/2021\n2019/2020\n2018/2019\n2017/2018\n2016/2017\n2015/20

In [ ]:
class="flex flex-col px-3 text-sm max-mm:px-0"

In [ ]:
class="eventRow flex w-full flex-col text-xs"

['Rugby Union\n/\nWorld\n/\nUnited Rugby Championship 2024/2025',
 '',
 'Rugby Union',
 '/',
 '/',
 "14 Jun 2025 - Play Offs\n1\nX\n2\nB's",
 '14 Jun 2025 - Play Offs',
 '14 Jun 2025 - Play Offs',
 "1\nX\n2\nB's",
 '1',
 'X',
 '2',
 "B's",
 '12:00\nLeinster\n32\n–\n7\nBulls\n-556\n+2800\n+392\n4',
 '12:00\nLeinster\n32\n–\n7\nBulls\n-556\n+2800\n+392\n4',
 '12:00\nLeinster\n32\n–\n7\nBulls',
 '12:00',
 '12:00',
 '12:00',
 'Leinster\n32\n–\n7\nBulls',
 'Leinster\n32\n–\n7\nBulls',
 'Leinster\n32\n–\n7\nBulls',
 'Leinster',
 '',
 '32\n–\n7',
 '32\n–\n7',
 '32',
 '7',
 'Bulls',
 '',
 '',
 '',
 '-556',
 '-556',
 '-556',
 '+2800',
 '+2800',
 '+2800',
 '+392',
 '+392',
 '+392',
 '4',
 '4',
 '4']

In [22]:
driver.close()#flex flex-col px-3 text-sm max-mm:px-0
driver.quit()

In [4]:
soup.find(class_='participant-name truncate')

In [24]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <link href="/favicon.ico" rel="icon"/>
  <meta content="m08AWek8xGTh3QkRJCDo8mjouPGzjfhocNLGMCiv" name="csrf-token"/>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="no-cache" http-equiv="pragma"/>
  <meta content="sport" property="og:type"/>
  <meta content="/images/logo.png" property="og:image"/>
  <meta content="Oddsportal.com" property="og:site_name"/>
  <meta content="LiveSport s.r.o." name="author"/>
  <meta content="(c) LiveSport s.r.o. 2007 - 2025" name="copyright"/>
  <meta content="index,follow" name="robots"/>
  <meta content="odds portal, odds comparison, archive, historical odds, results, betting odds, odds archive" name="keywords"/>
  <meta content="United Rugby Championship 2024/2025 results, scores &amp; historical odds, Rugby Union World. Don't miss on any United Rugby Championship 2024/2025

In [30]:
soup.prettify().find('__next')

-1

In [26]:
soup.find('div', attrs={'data-reactroot': True})

In [22]:
soup.find_all('script')

[<script>LUX=function(){function n(){return Date.now?Date.now():+new Date}var r,e=n(),t=window.performance||{},a=t.timing||{navigationStart:(null===(r=window.LUX)||void 0===r?void 0:r.ns)||e};function o(){return t.now?(r=t.now(),Math.floor(r)):n()-a.navigationStart;var r}(LUX=window.LUX||{}).ac=[],LUX.addData=function(n,r){return LUX.cmd(["addData",n,r])},LUX.cmd=function(n){return LUX.ac.push(n)},LUX.getDebug=function(){return[[e,0,[]]]},LUX.init=function(){return LUX.cmd(["init"])},LUX.mark=function(){for(var n=[],r=0;r<arguments.length;r++)n[r]=arguments[r];if(t.mark)return t.mark.apply(t,n);var e=n[0],a=n[1]||{};void 0===a.startTime&&(a.startTime=o());LUX.cmd(["mark",e,a])},LUX.markLoadTime=function(){return LUX.cmd(["markLoadTime",o()])},LUX.measure=function(){for(var n=[],r=0;r<arguments.length;r++)n[r]=arguments[r];if(t.measure)return t.measure.apply(t,n);var e,a=n[0],i=n[1],u=n[2];e="object"==typeof i?n[1]:{start:i,end:u};e.duration||e.end||(e.end=o());LUX.cmd(["measure",a,e])}

In [17]:
print(soup.prettify()[3000:])

';
   </script>
   <script>
    function switchVisibility(thisEl, selector) {
                const passwordField = document.getElementById(selector)
                if (passwordField.getAttribute('type') === 'password')  {
                    thisEl.classList.add('!bg-eye_icon')
                    thisEl.classList.remove('!bg-off_eye_icon')
                    passwordField.setAttribute('type', 'text')
                } else {
                    thisEl.classList.add('!bg-off_eye_icon')
                    thisEl.classList.remove('!bg-eye_icon')
                    passwordField.setAttribute('type', 'password')
                }
            }

            function addEventListenerByEl(thisEl) {
                const regModal = document.querySelectorAll(".regModal");
                
                if(regModal[0].classList.contains('flex')){
                    regModal[0].classList.remove('flex');
                    regModal[0].classList.add('hidden');
                } 
          

In [20]:
print(soup.prettify()[3900:])

move('flex');
                    regModal[0].classList.add('hidden');
                } 
                if(regModal[0].classList.contains('hidden')){
                    regModal[0].classList.remove('hidden');
                    regModal[0].classList.add('flex');
                }
            }
   </script>
   <script>
    window.dataLayer = window.dataLayer || [];
   </script>
   <script>
    document.addEventListener("DOMContentLoaded", () => {
                                (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
                    new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
                    j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
                    'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
                })(window,document,'script','dataLayer','GTM-W3GJZ5');
                            });
   </script>
   <style>
    *,::after,::before{border:0px solid;b

In [2]:
response.headers.get('Content-Type')

'text/html; charset=utf-8'

In [3]:
html = response.text.lower()

checks = {
    "cloudflare": any(k in html for k in ["cf-chl", "challenge-platform", "__cf_bm"]),
    "javascript_required": "enable javascript" in html or "checking your browser" in html,
    "react_shell": any(k in html for k in ["id=\"__next\"", "id=\"root\""]),
    "login_page": any(k in html for k in ["sign in", "log in", "password"]),
}

checks

{'cloudflare': False,
 'javascript_required': False,
 'react_shell': False,
 'login_page': True}

In [19]:
soup.find('p', string=re.compile('Connacht'))

In [20]:
soup.find('p', string='Connacht')

In [13]:
soup.prettify().find('Connacht')

77634

In [4]:
soup.prettify().find('odd-container-winning')

-1

In [21]:
soup.prettify()[77000:]

'-2012\\/results\\/"},"15":{"id":3532,"name":"2010\\/2011","url":"\\/rugby-union\\/world\\/magners-league-2010-2011\\/results\\/"},"16":{"id":1582,"name":"2009\\/2010","url":"\\/rugby-union\\/world\\/magners-league-2009-2010\\/results\\/"},"17":{"id":1581,"name":"2008\\/2009","url":"\\/rugby-union\\/world\\/magners-league-2008-2009\\/results\\/"},"default":115859,"active":115859},"oddsRequest":{"url":"\\/ajax-sport-country-tournament-archive_\\/8\\/EyHYm58U\\/","urlPartTz":-5,"urlPartQs":"?_="},"d":{"total":2,"rows":[{"id":8884491,"is-double":false,"superTemplate":{"id":"","name":"","url":"","shortUrl":""},"home":24499585,"away":24499587,"home-name":"Connacht","away-name":"Scarlets","home-country-two-chart-name":"ie","away-country-two-chart-name":"207","home-participant-id":1,"away-participant-id":1,"status-id":1,"event-stage-id":1,"event-stage-name":"Scheduled","event-stage-name-short":"","tournament_id":115859,"tournament-stage-id":191975,"tournament-stage-type-id":2,"tournament-stag

In [3]:
soup.prettify()

'<!DOCTYPE html>\n<html lang="en">\n <head>\n  <meta charset="utf-8"/>\n  <meta content="width=device-width, initial-scale=1" name="viewport"/>\n  <link href="/favicon.ico" rel="icon"/>\n  <meta content="FNpmgz7zJCWanFYScbJ1P6Wg2eVThZM2wNR1Ad0M" name="csrf-token"/>\n  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>\n  <meta content="no-cache" http-equiv="pragma"/>\n  <meta content="sport" property="og:type"/>\n  <meta content="/images/logo.png" property="og:image"/>\n  <meta content="Oddsportal.com" property="og:site_name"/>\n  <meta content="LiveSport s.r.o." name="author"/>\n  <meta content="(c) LiveSport s.r.o. 2007 - 2025" name="copyright"/>\n  <meta content="index,follow" name="robots"/>\n  <meta content="odds portal, odds comparison, archive, historical odds, results, betting odds, odds archive" name="keywords"/>\n  <meta content="United Rugby Championship 2024/2025 results, scores &amp; historical odds, Rugby Union World. Don\'t miss on any United Rugby Cham

In [11]:
result_container = soup.find_all(class_='min-h-[80vh]')
result_container

[<div class="min-h-[80vh]">
 <next-matches :data-context='"results"' :leagues-button='{"MyLeagueButton":{"id":115859,"name":"United Rugby Championship","countryId":8,"addText":"Add \"United Rugby Championship\" to My Leagues","removeText":"Remove \"United Rugby Championship\" from My Leagues"},"myLeaguesAction":"add"}' :odds-request='{"url":"\/ajax-sport-country-tournament-archive_\/8\/EyHYm58U\/","urlPartTz":-5,"urlPartQs":"?_="}' :page-name="'tournament-results'">
 </next-matches>
 </div>]

In [9]:
result_container.find(class_='group flex')

In [6]:
[i.text for i in soup.find_all('p')]

['football',
 'tennis',
 'basketball',
 'other',
 'popular now',
 '\n\n                    Oddsportal\n                \n',
 'partners']